In [1]:
# ==========================================
# ECG IMAGE GENERATION (OOM SAFE VERSION)
# ==========================================

import os
import wfdb
import pandas as pd
from tqdm import tqdm
import shutil
import numpy as np
import gc # Added for explicit garbage collection

# 🔥 CRITICAL FIX 1: Set Matplotlib to a non-interactive backend
# This must be done BEFORE importing pyplot. It stops Matplotlib from 
# trying to render anything to the VS Code UI, drastically saving RAM.
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# ==========================================
# PATHS
# ==========================================
BASE_PATH = "data/ptb-xl"
CSV_PATH = os.path.join(BASE_PATH, "ptbxl_database.csv")
SAVE_PATH = os.path.join(BASE_PATH, "images")

# ==========================================
# CLEAN OLD IMAGES
# ==========================================
if os.path.exists(SAVE_PATH):
    shutil.rmtree(SAVE_PATH)

os.makedirs(SAVE_PATH, exist_ok=True)

# ==========================================
# LOAD DATA
# ==========================================
df = pd.read_csv(CSV_PATH)
print("Total records:", len(df))

# ==========================================
# GENERATE IMAGES
# ==========================================
for i, row in tqdm(df.iterrows(), total=len(df)):
    try:
        record_path = os.path.join(BASE_PATH, row['filename_lr'])

        # Read ECG signal
        signal, meta = wfdb.rdsamp(record_path)

        # ==========================================
        # CREATE FIGURE
        # ==========================================
        fig, axes = plt.subplots(6, 2, figsize=(12, 8))
        axes = axes.flatten()

        for lead in range(12):
            # Normalize signal
            sig = signal[:, lead]
            sig = (sig - np.mean(sig)) / (np.std(sig) + 1e-8)

            # Plot
            axes[lead].plot(sig, linewidth=1.2, color='black')
            axes[lead].set_facecolor('#f5f5f5')
            axes[lead].grid(True, linestyle='--', linewidth=0.5)

            # Titles and clean axes
            axes[lead].set_title(f"Lead {lead+1}", fontsize=8)
            axes[lead].set_xticks([])
            axes[lead].set_yticks([])

        plt.tight_layout()

        # Save image
        img_file = os.path.join(SAVE_PATH, f"{row['ecg_id']}.png")
        fig.savefig(img_file, dpi=150)

        # 🔥 CRITICAL FIX 2: Aggressive cleanup
        fig.clf()           # Clear the current figure
        plt.close('all')    # Close all figures to ensure nothing lingers

        # 🔥 CRITICAL FIX 3: Force garbage collection every 100 iterations
        if i % 100 == 0:
            gc.collect()

    except Exception as e:
        # It's good practice to print the error so you know if data is failing
        # print(f"Error on row {i}: {e}") 
        continue

print("✅ ECG images generated successfully!")

Total records: 21799


100%|██████████| 21799/21799 [1:46:17<00:00,  3.42it/s]

✅ ECG images generated successfully!
